# Notebook 1 — Data validation & extraction for final BERTopic model (taxonomy + Radway)

This notebook is the **entry point** for Stage 10 analysis. It:
- loads the final BERTopic model with taxonomy & Radway mappings
- merges Stage 08 label metadata (label, scene summary, category tags)
- exports a **topic-level lookup table** used downstream to aggregate book-level proportions
- runs **QA checks** (missing mappings, keyword quality, confidence distribution)

Outputs are written to: `results/stage10_correlation_analysis/taxonomy_radway_eda/`


## 1) Setup & paths (keep consistent with your existing notebook)
This reproduces the same path conventions used in `radway_model_interactive_eda.ipynb`.

In [1]:
# --- Robust project_root fix: handle not defined and relative/invalid cases safely ---
from pathlib import Path

def safe_project_root(project_root_var=None) -> Path:
    """Ensure project_root is defined, exists, and is absolute.
    Attempts to infer sensible location if not provided or not valid."""
    # 1. Use arg if present, else try global, else fallback to cwd scan
    _pr = project_root_var
    try:
        if _pr is None:
            _pr = globals().get("project_root", None)
        # If still None or empty, try environment
        if _pr in (None, ""):
            _pr = Path.home()  # fallback to home, for now
        else:
            _pr = Path(_pr)
    except Exception:
        _pr = Path.cwd()
    
    _pr = _pr.expanduser().resolve()
    # If not valid, try scanning for src/results
    SEARCH_MARKERS = ["src", "results"]
    def looks_like_root(p):
        return all((p/pth).exists() for pth in SEARCH_MARKERS)
    
    # If path is ".", doesn't exist, or doesn't have src/results, look up tree
    if not _pr.exists() or _pr == Path(".") or not looks_like_root(_pr):
        # 1. Check up from cwd
        for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
            if looks_like_root(p):
                _pr = p
                break
        else:
            # 2. Fallback to hardcoded known path (edit if needed)
            KNOWN = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
            if looks_like_root(KNOWN):
                _pr = KNOWN.resolve()
            else:
                raise RuntimeError("Could not determine a valid project_root!")
    return _pr

project_root = safe_project_root()
print(f"✓ Project root: {project_root}")

✓ Project root: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor


In [ ]:
from __future__ import annotations

import os
import json
import ast
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd

# Optional: model loading (only needed if you run this inside the project repo)
try:
    from bertopic import BERTopic
except Exception:
    BERTopic = None  # type: ignore

# ---- Project root detection ----
# Use project_root from previous cell if available, otherwise find it
if 'project_root' not in globals() or project_root is None:
    def find_project_root(start: Path | None = None) -> Path:
        """Find repo root by scanning parent dirs for `src/` (and optionally `results/`)."""
        start = (start or Path.cwd()).resolve()
        for p in [start, *start.parents]:
            if (p / "src").exists():
                return p
        return start
    
    project_root = Path(os.environ.get("PROJECT_ROOT", "")).expanduser()
    project_root = project_root if project_root.exists() else find_project_root()

print(f"Project root: {project_root}")

# ---- Load defaults from your project (if available) ----
try:
    from src.stage06_topic_exploration.explore_retrained_model import (
        DEFAULT_BASE_DIR,
        DEFAULT_EMBEDDING_MODEL,
    )
except Exception:
    # Fallbacks (edit if running outside the repo)
    DEFAULT_BASE_DIR = project_root / "models"
    DEFAULT_EMBEDDING_MODEL = "paraphrase-MiniLM-L6-v2"

print(f"DEFAULT_BASE_DIR: {DEFAULT_BASE_DIR}")
print(f"DEFAULT_EMBEDDING_MODEL: {DEFAULT_EMBEDDING_MODEL}")

# ---- Paths (match your existing notebook) ----
stage_subfolder = "stage09_category_mapping"
model_suffix = "_with_radway_mappings"

MODEL_PATH = DEFAULT_BASE_DIR / DEFAULT_EMBEDDING_MODEL / stage_subfolder / f"model_1{model_suffix}"

LABELS_FILENAME = (
    "labels_pos_openrouter_mistralai_Mistral-Nemo-Instruct-2407_"  # adjust if needed
    "romance_aware_paraphrase-MiniLM-L6-v2.json"
)
LABELS_PATH = project_root / "results" / "stage08_llm_labeling" / LABELS_FILENAME

OUT_DIR = project_root / "results" / "stage10_correlation_analysis" / "taxonomy_radway_eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_PATH:  {MODEL_PATH}")
print(f"LABELS_PATH: {LABELS_PATH}")
print(f"OUT_DIR:     {OUT_DIR}")


## 2) Load Stage 08 labels JSON (topic label + scene summary + tags)
This is the preferred source for `label`, `scene_summary`, and category tags.

In [ ]:
def load_labels_metadata(labels_path: Path) -> dict[int, dict[str, Any]]:
    """Load full metadata from Stage 08 labels JSON.
    Accepts both:
      - rich JSON per topic (dict with label, scene_summary, etc.)
      - simple mapping {topic_id: "label"}
    """
    # Convert to Path if needed
    labels_path = Path(labels_path)
    
    # If path doesn't exist, try to find project root and reconstruct path
    if not labels_path.exists():
        # Find project root by looking for src/ and results/ directories
        from pathlib import Path as P
        cwd = P.cwd()
        project_root = None
        for parent in [cwd, *cwd.parents]:
            if (parent / "src").exists() and (parent / "results").exists():
                project_root = parent.resolve()
                break
        
        if project_root:
            # Extract the path components after project root
            # The labels_path should be: project_root / "results" / "stage08_llm_labeling" / filename
            path_str = str(labels_path)
            if "results" in path_str and "stage08_llm_labeling" in path_str:
                # Reconstruct: project_root / results / stage08_llm_labeling / filename
                filename = labels_path.name
                labels_path = project_root / "results" / "stage08_llm_labeling" / filename
    
    if not labels_path.exists():
        # Fall back: try to find any labels_*.json in the same folder
        labels_dir = labels_path.parent
        
        if not labels_dir.exists():
            raise FileNotFoundError(
                f"Labels directory does not exist: {labels_dir}"
            )
        
        # Try non-recursive glob first (files directly in the directory)
        candidates = sorted(labels_dir.glob("labels_*.json"))
        
        # If no files found, try recursive search (including subdirectories)
        if not candidates:
            candidates = sorted(labels_dir.rglob("labels_*.json"))
        
        if not candidates:
            # Provide helpful error message with available files
            available_files = [f.name for f in labels_dir.iterdir() if f.is_file()]
            raise FileNotFoundError(
                f"No labels file found at {labels_path}\n"
                f"Directory exists: {labels_dir}\n"
                f"Available files (first 10): {available_files[:10]}"
            )
        
        # Use the newest file (last in sorted list)
        labels_path = candidates[-1]
        print(f"⚠️ LABELS_PATH not found; using newest candidate: {labels_path.name}")

    print(f"Loading labels metadata from: {labels_path}")
    with open(labels_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    metadata: dict[int, dict[str, Any]] = {}
    for topic_id_str, topic_data in data.items():
        topic_id = int(topic_id_str)
        if isinstance(topic_data, dict):
            metadata[topic_id] = topic_data.copy()
        else:
            metadata[topic_id] = {"label": str(topic_data)}

    print(f"✓ Loaded Stage 08 label metadata for {len(metadata)} topics")
    return metadata

labels_metadata = load_labels_metadata(LABELS_PATH)
list(labels_metadata.items())[:2]


## 3) Load model (or fall back to previously exported `full_model_data.csv`)
If you run this inside the repo and the model exists, it loads the BERTopic model.
If not, it loads the last exported topic table from `OUT_DIR/full_model_data.csv`.

In [ ]:
def load_model(model_path: Path):
    if BERTopic is None:
        raise ImportError("BERTopic is not available in this environment.")
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found at: {model_path}")
    return BERTopic.load(str(model_path))

# Fix: Ensure OUT_DIR and all path variables match the expected absolute paths.
FULL_CSV = OUT_DIR / "full_model_data.csv"
ARCHIVE_CSV = OUT_DIR / "archive" / "full_model_data.csv"
ARCHIVE_PARQUET = OUT_DIR / "archive" / "full_model_data.parquet"

# Print the resolved absolute paths to avoid confusion
print(f"FULL_CSV: {FULL_CSV.resolve()}")
print(f"ARCHIVE_CSV: {ARCHIVE_CSV.resolve()}")
print(f"ARCHIVE_PARQUET: {ARCHIVE_PARQUET.resolve()}")
print(f"MODEL_PATH: {MODEL_PATH.resolve()}")

# Additionally check alternate model directory (if present)
RETRAINED_MODEL_PATH = Path(
    "/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/retrained/paraphrase-MiniLM-L6-v2/stage09_category_mapping/model_1_with_radway_mappings"
)
model = None

tried_model_paths = [MODEL_PATH]
if RETRAINED_MODEL_PATH.exists():
    tried_model_paths.insert(0, RETRAINED_MODEL_PATH)

model_loaded = False
for candidate_model_path in tried_model_paths:
    if candidate_model_path.exists() and BERTopic is not None:
        print(f"🔍 Trying to load BERTopic model from: {candidate_model_path}")
        model = load_model(candidate_model_path)
        print(f"✓ Model loaded from: {candidate_model_path}")
        model_loaded = True
        break

if not model_loaded:
    print("⚠️ Model not available at the checked locations; will use exported CSV if present.")
    # Print which files exist for debugging
    for file_path in [FULL_CSV, ARCHIVE_CSV, ARCHIVE_PARQUET]:
        print(f"Exists ({file_path}): {file_path.exists()}")

    if not FULL_CSV.exists() and not ARCHIVE_CSV.exists() and not ARCHIVE_PARQUET.exists():
        print(f"   Expected: {FULL_CSV.resolve()}")
        print(f"        or: {ARCHIVE_CSV.resolve()}")
        print(f"        or: {ARCHIVE_PARQUET.resolve()}")

## 4) Extract topic-level table
This table is the *topic lookup* you will later join with `book_topic_probs.parquet` and `chapter_topic_probs.parquet` (outputs from `generate_topic_probabilities.py` in `results/stage10_correlation_analysis/`).

In [ ]:
def extract_all_fields(
    model,
    labels_metadata: Optional[dict[int, dict[str, Any]]] = None,
) -> pd.DataFrame:
    rows = []

    # topic ids (exclude -1 outlier)
    topic_ids = [tid for tid in getattr(model, "topic_representations_", {}).keys() if tid != -1]

    for topic_id in sorted(topic_ids):
        row: dict[str, Any] = {"topic_id": topic_id}

        # Keywords
        if hasattr(model, "topic_representations_") and topic_id in model.topic_representations_:
            kws = model.topic_representations_[topic_id]
            row["keywords"] = ", ".join([kw[0] for kw in kws[:10]])
            row["num_keywords"] = len(kws)
            row["all_keywords"] = [kw[0] for kw in kws]

        # Stage 08 label metadata (preferred)
        if labels_metadata and topic_id in labels_metadata:
            meta = labels_metadata[topic_id]
            row["label"] = meta.get("label")
            row["scene_summary"] = meta.get("scene_summary")
            row["primary_categories"] = ", ".join(meta.get("primary_categories", [])) or None
            row["secondary_categories"] = ", ".join(meta.get("secondary_categories", [])) or None
            row["label_is_noise"] = bool(meta.get("is_noise", False))
            row["label_rationale"] = meta.get("rationale")
        else:
            row["label"] = getattr(model, "topic_labels_", {}).get(topic_id)
            row["scene_summary"] = None
            row["primary_categories"] = None
            row["secondary_categories"] = None
            row["label_is_noise"] = None
            row["label_rationale"] = None

        # Taxonomy (Stage 2)
        tax = getattr(model, "topic_taxonomy_", {}).get(topic_id, {})
        row.update({
            "taxonomy_main_id": tax.get("main_category_id"),
            "taxonomy_main_name": tax.get("main_category_name"),
            "taxonomy_main_group": tax.get("main_category_group"),
            "taxonomy_secondary_id": tax.get("secondary_category_id"),
            "taxonomy_secondary_name": tax.get("secondary_category_name"),
            "taxonomy_secondary_group": tax.get("secondary_category_group"),
            "taxonomy_confidence": tax.get("confidence"),
            "taxonomy_is_noise": bool(tax.get("is_noise", False)),
        })

        # Radway (Stage 3)
        rad = getattr(model, "topic_radway_", {}).get(topic_id, {})
        row.update({
            "radway_main_id": rad.get("radway_main_id"),
            "radway_main_name": rad.get("radway_main_name"),
            "radway_secondary_id": rad.get("radway_secondary_id"),
            "radway_phase": rad.get("radway_phase") if rad.get("radway_phase") is not None else "NA",
            "radway_phase_name": rad.get("radway_phase_name"),
            "radway_is_none": bool(rad.get("radway_is_none", False)),
            "radway_confidence": rad.get("radway_confidence"),
            "radway_rationale": rad.get("radway_rationale"),
        })

        rows.append(row)

    return pd.DataFrame(rows)

if model is not None:
    df_topics = extract_all_fields(model, labels_metadata=labels_metadata)
else:
    # fallback: load exported
    df_topics = pd.read_csv(FULL_CSV)
    print(f"✓ Loaded exported table: {FULL_CSV} ({df_topics.shape[0]} rows)")

df_topics.shape


## 5) QA checks (Notebook 1 should keep these)
These checks prevent silent downstream errors when you aggregate to book-level.

In [ ]:
# Basic schema sanity
expected_cols = {
    "topic_id","label","taxonomy_main_id","taxonomy_main_name","taxonomy_main_group",
    "radway_main_id","radway_main_name","radway_phase_name","radway_is_none"
}
missing = expected_cols - set(df_topics.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

# Duplicates
dup = df_topics["topic_id"].duplicated().sum()
print(f"Duplicate topic_id rows: {dup}")

# Missing mappings
n_total = len(df_topics)
n_tax_missing = df_topics["taxonomy_main_id"].isna().sum()
n_rad_missing = df_topics["radway_main_id"].isna().sum()
print(f"Taxonomy missing: {n_tax_missing}/{n_total}")
print(f"Radway missing:   {n_rad_missing}/{n_total}")

# Radway none vs function
rad_none = (df_topics["radway_is_none"] == True).sum()
rad_fn = (df_topics["radway_is_none"] == False).sum()
print(f"Radway function topics: {rad_fn}")
print(f"Radway none topics:     {rad_none}")

# Keyword-quality check: many empty tokens often indicate artifacts
def empty_ratio(all_keywords_cell) -> float:
    # all_keywords might be a list (fresh extract) or a string (CSV)
    if isinstance(all_keywords_cell, str):
        try:
            lst = ast.literal_eval(all_keywords_cell)
        except Exception:
            return np.nan
    else:
        lst = all_keywords_cell
    if not lst:
        return np.nan
    return sum(1 for w in lst if not w) / len(lst)

df_topics["_empty_kw_ratio"] = df_topics["all_keywords"].apply(empty_ratio)
bad_kw = df_topics[df_topics["_empty_kw_ratio"] > 0.5].copy()

print(f"Topics with >50% empty keywords: {len(bad_kw)}")
bad_kw[["topic_id","label","taxonomy_main_name","radway_main_id","_empty_kw_ratio"]].head(20)


In [ ]:
# Save a 'needs review' list for manual inspection
needs_review = df_topics[
    df_topics["taxonomy_main_id"].isna()
    | df_topics["radway_main_id"].isna()
    | (df_topics["_empty_kw_ratio"] > 0.5)
][["topic_id","label","keywords","taxonomy_main_name","taxonomy_main_group","radway_main_id","radway_phase_name","taxonomy_confidence","radway_confidence","_empty_kw_ratio"]]

needs_review_path = OUT_DIR / "topics_needs_review.csv"
needs_review.to_csv(needs_review_path, index=False)
print(f"✓ Wrote: {needs_review_path}  (n={len(needs_review)})")


## 6) Export topic lookup + summary tables
These outputs are used downstream to build book-level category proportions and indices.

In [ ]:
# Summary statistics (same fields as your prior notebook)
summary = {
    "total_topics": int(len(df_topics)),
    "topics_with_labels": int(df_topics["label"].notna().sum()),
    "topics_with_taxonomy": int(df_topics["taxonomy_main_id"].notna().sum()),
    "topics_with_radway": int(df_topics["radway_main_id"].notna().sum()),
    "topics_with_radway_function": int((df_topics["radway_is_none"] == False).sum()),
    "topics_with_radway_none": int((df_topics["radway_is_none"] == True).sum()),
    "unique_taxonomy_categories": int(df_topics["taxonomy_main_name"].nunique(dropna=True)),
    "unique_taxonomy_groups": int(df_topics["taxonomy_main_group"].nunique(dropna=True)),
    "unique_radway_functions": int(df_topics.loc[df_topics["radway_is_none"] == False, "radway_main_name"].nunique(dropna=True)),
    "unique_radway_phases": int(df_topics["radway_phase_name"].nunique(dropna=True)),
}
summary_path = OUT_DIR / "summary_statistics.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"✓ Wrote: {summary_path}")

        # Topic lookup (for merging with book_topic_probs.parquet later)
topic_lookup = df_topics[[
    "topic_id",
    "taxonomy_main_id","taxonomy_main_name","taxonomy_main_group",
    "taxonomy_secondary_id","taxonomy_secondary_name","taxonomy_secondary_group",
    "taxonomy_confidence","taxonomy_is_noise",
    "radway_main_id","radway_main_name","radway_phase","radway_phase_name","radway_is_none","radway_confidence",
    "label","scene_summary","primary_categories","secondary_categories","label_is_noise",
]].copy()

topic_lookup_path = OUT_DIR / "topic_lookup.parquet"
topic_lookup.to_parquet(topic_lookup_path, index=False)
print(f"✓ Wrote: {topic_lookup_path}")

# Also export full table (CSV + Parquet) to keep parity with your existing notebook
full_csv = OUT_DIR / "full_model_data.csv"
full_parquet = OUT_DIR / "full_model_data.parquet"
df_topics.to_csv(full_csv, index=False)
df_topics.to_parquet(full_parquet, index=False)
print(f"✓ Wrote: {full_csv}")
print(f"✓ Wrote: {full_parquet}")

# Cross-tabs to reuse in EDA notebook (Notebook 3), but stored here for convenience
ct_group_phase = pd.crosstab(df_topics["taxonomy_main_group"], df_topics["radway_phase_name"])
ct_group_phase.to_csv(OUT_DIR / "crosstab_taxonomy_group_x_radway_phase.csv")

ct_top = pd.crosstab(df_topics["taxonomy_main_name"], df_topics["radway_main_id"])
ct_top.to_csv(OUT_DIR / "crosstab_taxonomy_main_x_radway_id.csv")

print("✓ Wrote crosstab CSVs")


## What comes next
- **Notebook 2** will merge `topic_lookup.parquet` with `book_topic_probs.parquet` / `chapter_topic_probs.parquet` (from `results/stage10_correlation_analysis/`) and produce book-level category proportions.
- **Notebook 3** will use those outputs for EDA (heatmaps, volcano plots, correlations, clustering).
